[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://githubtocolab.com/Soliman2020/python_48h_notebooks/blob/master/Session_05_Expansion_Libraries-and-small-test-cases.ipynb)

# Concept 3 expansion

**Libraries and small test cases**

### Importing a module: four ways

`import` brings existing code into your notebook. This subsection shows the four forms you will meet,
so that the short forms are never mysterious.

- `import math` — then use `math.sqrt(...)`. Clearest in a long program.
- `from math import sqrt` — use `sqrt(...)` directly. Shorter, but the name no longer shows where it
  came from, and it can clash with a name you define yourself.
- `import statistics as stats` — a short alias. `stats` is a common convention for `statistics`.
- `from datetime import datetime, timedelta` — several names at once when you need them together.

In [ ]:
# 1. Import the module, then use module.name
import math
print(math.sqrt(81))

# 2. Import one name directly - shorter, but less obvious where it came from
from math import sqrt
print(sqrt(81))

# 3. Import with an alias
import statistics as stats
print(stats.mean([20, 21, 22, 30]))

# 4. Import several names at once
from datetime import datetime, timedelta
print(datetime(2026, 10, 18, 8, 30))

### `math` — numbers beyond the operators

`math` covers what `+ - * /` cannot: square roots, rounding to whole numbers, constants and
accurate comparison of decimal values.

In [ ]:
import math

print(math.sqrt(144))                     # square root
print(math.floor(7.9), math.ceil(7.1))    # round down / round up to a whole number
print(round(math.pi, 4))                  # pi, to four decimal places
print(math.pow(2, 10))                    # 2 to the power of 10

# Comparing decimal values needs care: 0.1 + 0.2 is not exactly 0.3 in binary
print(0.1 + 0.2 == 0.3)
print(math.isclose(0.1 + 0.2, 0.3))

**Stop and think:** In Session 01 we tested values with `==`. Why is that unreliable for decimal
numbers, and what does `math.isclose` do instead? What would `0.145 * 200` need before you could
compare it with `29.0`?

**Watch for:** form 2 above imported only `sqrt`. If a later cell defines its own `sqrt` function, it
replaces the imported one — one more reason the full `import math` form is safer in a long notebook.

### `statistics` — the rest of the summary

The cell above printed the **mean** and the **median** of `readings`. The same module also reports the
most frequent value and how spread out the values are.

In [ ]:
import statistics

readings = [20, 21, 22, 30]

print('Mode  :', statistics.mode([1, 2, 2, 3, 4]))        # most frequent value
print('Stdev :', round(statistics.stdev(readings), 3))    # spread of the values
print('Range :', max(readings) - min(readings))           # highest minus lowest

# Mean and median, for comparison with the stdev above
print('Mean  :', statistics.mean(readings))
print('Median:', statistics.median(readings))

### What happens with an empty list?

This is the question in the **Stop and think** prompt above. `statistics.mean` refuses to guess: it
raises an error instead of returning `0` or `None`.

In [ ]:
import statistics

readings = []          # imagine a sensor that sent nothing today

try:
    print(statistics.mean(readings))
except statistics.StatisticsError as error:
    print('Cannot summarise:', error)

# A guard clause is usually simpler than catching the error
if readings:
    print('Mean:', statistics.mean(readings))
else:
    print('No readings to summarise')

**Industry transfer:** Deciding what a function should do with empty or missing input is a real
design decision — refuse loudly, or return a documented default. Which behaviour would your audit,
laboratory or monitoring work need, and why?

**Looking ahead:** the `try` / `except` block is the topic of **Session 06 — Errors, Debugging & Small
Tests**. For now, notice that the error message tells you exactly what was wrong.

### `datetime` — dates, times and the difference between them

Subtracting two `datetime` values gives a `timedelta`. That is how you turn "raised at" and "fixed
at" into the `hours_open` field used in the record examples of Sessions 03 and 04.

In [ ]:
from datetime import datetime, timedelta

raised_at = datetime(2026, 10, 18, 8, 30)
fixed_at  = datetime(2026, 10, 19, 14, 0)

gap = fixed_at - raised_at

print('Gap        :', gap)
print('Hours open :', gap.total_seconds() / 3600)
print('Whole days :', gap.days)
print('Fixed date :', fixed_at.strftime('%d %b %Y'))
print('Review due :', (fixed_at + timedelta(days=3)).date())

**Stop and think:** the gap is `1 day, 5:30:00`, so why is `gap.days` alone not enough? Where
would you use `datetime.now()` in this notebook — and why would a test case use fixed dates such as
`datetime(2026, 10, 18, 8, 30)` instead?

### `json` — records that survive outside the notebook

A dictionary lives only while the notebook is running. `json` turns it into text, so it can be saved
in a file, sent to another program, or read back tomorrow — in Python or in any other language.

In [ ]:
import json

case = {'case_id': 'C-01', 'priority': 'high', 'hours': 9, 'open': True}

as_text = json.dumps(case, sort_keys=True)   # dict -> text
print(as_text)
print(type(as_text).__name__)

back = json.loads(as_text)                    # text -> dict
print(back['case_id'], back['hours'])
print(back == case)

**Stop and think:** the text shows `true`, not Python's `True`. Why must a file format avoid
Python's own spelling of its values? **Looking ahead:** Session 07 reads and writes these files.

### `pathlib` — writing and reading a small file

`Path` builds file paths without worrying about the operating system. The report below lands in
`OUTPUT_DIR`, the `outputs` folder created by the setup cell — the same folder the optional download
cell at the end of the notebook packs into a ZIP.

In [ ]:
from pathlib import Path
import json

report_path = OUTPUT_DIR / 's05_report.json'

report_path.write_text(json.dumps({'mean': 23.25, 'count': 4}), encoding='utf-8')
print(report_path.name)
print(report_path.exists())
print(report_path.read_text(encoding='utf-8'))

saved = json.loads(report_path.read_text(encoding='utf-8'))
print('Read back count:', saved['count'])

**Stop and think:** what would change if the file already existed and contained yesterday's
report — does `write_text` add to it or replace it? (Change the dictionary, run the cell twice and
watch the content.)

**Industry transfer:** audit evidence, laboratory results and monitoring exports all end up as files.
The pattern is always the same three steps: build the record, write it, read it back to check.

### `assert` — checking your own assumptions

An `assert` states what must be true. If it is, nothing happens and the notebook continues. If it is
not, Python stops with an `AssertionError` and shows your message — usually the fastest way to find
the line where your thinking went wrong.

In [ ]:
def format_case(case_id, status):
    return f'{case_id}: {status.upper()}'

# A check that passes: the message is only shown when the check fails
assert format_case('C-01', 'open') == 'C-01: OPEN', 'format_case returned the wrong text'
print('Check 1 passed')

# The same check with a wrong expectation, so you can see a failing assert
try:
    assert format_case('C-01', 'open') == 'C-01: closed', 'format_case returned the wrong text'
except AssertionError as error:
    print('Check 2 failed:', error)

**Stop and think:** how many lines of the notebook run after a failing `assert`? (Put an
`assert False` in a scratch cell and see.)

**Watch for:** `assert` is a learning and development tool, not a test suite. It is removed entirely
when Python runs with the `-O` flag, which is why the notebook says it "does not replace formal
production testing".

### Small test cases: check one function with several inputs

One `assert` checks one case. A short loop of cases is more useful: it shows a PASS or FAIL line for
every input you care about, including the boundary values.

In [ ]:
def is_overdue(hours, limit=24):
    """Return True when hours exceeds the allowed limit."""
    return hours > limit

def check(label, actual, expected):
    """Print PASS or FAIL for one test case."""
    result = 'PASS' if actual == expected else 'FAIL'
    print(f'{result}  {label}: expected {expected}, got {actual}')

# Boundary values are the interesting ones: just over, exactly on, and well below
check('30 hours, limit 24', is_overdue(30), True)
check('25 hours, limit 24', is_overdue(25), True)
check('24 hours, limit 24', is_overdue(24), False)   # exactly on the limit
check('30 hours, limit 48', is_overdue(30, 48), False)

# A deliberately wrong expectation, so you can see a FAIL line
check('deliberate mistake', is_overdue(20), True)

**Stop and think:** which case above is the boundary case, and why is it the one most likely to
catch a mistake? (`is_overdue` uses `>`, so 24 hours is *not* overdue. Would your organisation agree?)

**Looking ahead:** writing a few cases like this around every function you write is the habit that
Session 06 builds on.

### Exploring a module you have never used

You do not need to memorise a module. Ask it what it contains, and read the documentation of anything
that looks useful.

In [ ]:
import statistics

# dir() lists the names inside a module
print('mean' in dir(statistics), 'mode' in dir(statistics), 'median' in dir(statistics))

# A docstring is documentation you can read back at any time
def escalate(hours, limit=24):
    """Return True when an incident should be escalated."""
    return hours > limit

print(escalate.__doc__)

# help() prints the full documentation of a module, a function or your own code.
# Uncomment either line and run the cell to read it.
# help(statistics.mean)
# help(escalate)

**Industry transfer:** the standard library already contains statistics, date arithmetic, JSON
handling and file paths. Before writing your own version of something, check whether it is already
there — that is what "reuse" means in this session's title.